<a href="https://colab.research.google.com/github/hazelkimhyejin/DEMO/blob/main/CT2011_Lab_05_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Predicting Miles per gallon (mpg)**
The data concerns city-cycle fuel consumption in miles per gallon, to be predicted in terms some multivalued discrete and continuous features.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.simplefilter(action='ignore')
# read the relevant file
path = 'https://raw.githubusercontent.com/timcyku/ct2011/refs/heads/main/auto-mpg.csv'

df = pd.read_csv(path)

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   year          398 non-null    int64  
 7   origin        398 non-null    int64  
dtypes: float64(4), int64(4)
memory usage: 25.0 KB


In [3]:
# only few rows under "horsepower" have missing data
# drop these rows from the dataset
df = df.dropna(subset=['horsepower'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 392 entries, 0 to 397
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           392 non-null    float64
 1   cylinders     392 non-null    int64  
 2   displacement  392 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        392 non-null    int64  
 5   acceleration  392 non-null    float64
 6   year          392 non-null    int64  
 7   origin        392 non-null    int64  
dtypes: float64(4), int64(4)
memory usage: 27.6 KB


In [4]:
# Declare feature vector and target variable
X = df.drop(['mpg'], axis =1) #feature
y = df['mpg'] #target

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.25, random_state = 0)

In [6]:
# display categorical variables
categorical = [col for col in X_train.columns if X_train[col].dtypes == 'O']
categorical

[]

In [7]:
# display numerical variables

numerical = [col for col in X_train.columns if X_train[col].dtypes != 'O']
numerical

['cylinders',
 'displacement',
 'horsepower',
 'weight',
 'acceleration',
 'year',
 'origin']

# **Task 1: Model Fitting**

Use the pipeline approach in Lab 05 Worksheet. However, instead of **DecisionTreeClassifier**, use **DecisionTreeRegressor** since the target variable "mpg" is a continuous variable NOT categorical variable.

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

In [9]:
# Preprocessing: scale numeric + one-hot encode categorical
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical)
])

In [10]:
# Pipeline: preprocessing + model
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('dtr', DecisionTreeRegressor(random_state=0))
])


In [15]:
# Fit the training data
# Task 1 Answer:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['cylinders', 'displacement',
                                                   'horsepower', 'weight',
                                                   'acceleration', 'year',
                                                   'origin']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  [])])),
                ('dtr', DecisionTreeRegressor(random_state=0))])

# **Task 2: Make predictions on X_test and evaluate**

In [20]:
# Predict mpg values on X_test
# Task 2 Answer:
y_test_pred = pipeline.predict(X_test)

In [21]:
# the score method is the R-squared for continuous target variable
pipeline.score(X_test, y_test)

0.748663439112374

In [22]:
# Double check this is indeed R-squared score
from sklearn.metrics import r2_score
r2_score(y_test, y_test_pred)

0.748663439112374

# **Task 3: Tuning Hyperparameters using GridSearchCV**

In [23]:
size = X_train.shape[0]
num_features = preprocessor.fit_transform(X_train).shape[1]

In [25]:
# By trial and error, change the values of the following
# hyperparameter until you can a score (R2) higher than the one in Task 2
# Task 3 Answer:

param_grid = {
    'dtr__max_depth': [10, 20, 30, None],  # Expanded range, including None for no limit
    'dtr__max_leaf_nodes': [50, 100, 150],  # Adjusted for more flexibility
    'dtr__min_samples_leaf': [1, 5, size//20, size//15]  # Smaller values for finer control
}

# Perform grid search with cross-validation
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Print the best parameters and the corresponding R-squared score on the test set
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation R-squared score:", grid_search.best_score_)
print("Test set R-squared score:", grid_search.score(X_test, y_test))

# Store the best model's predictions for further evaluation if needed
y_pred_best = grid_search.predict(X_test)



Best parameters: {'dtr__max_depth': 10, 'dtr__max_leaf_nodes': 50, 'dtr__min_samples_leaf': 5}
Best cross-validation R-squared score: 0.805202577148782
Test set R-squared score: 0.7793632935411237
